# Extracting YouTube Data with the YouTube Data API v3

This notebook accompanies a research seminar on the extraction of YouTube data. It shows how to use the YouTube Data API v3 to retrieve videos from one channel and collect their top-level comments. It is designed for a guided demonstration in Google Colab and uses a small default sample so that the complete workflow can be shown within the presentation.

## Research workflow

1. Identify the uploads playlist associated with a channel.
2. Retrieve the video identifiers published in a selected period.
3. Request video metadata and filter by duration.
4. Retrieve top-level comments through paginated API requests.
5. Save videos, comments, users, and an extraction log as CSV files.

> **Scope:** the notebook covers data extraction and export only. It does not perform community detection, hate-speech classification, sentiment analysis, emotion analysis, or any other form of data analysis. It retrieves top-level comments, not replies to those comments. Deleted, private, unavailable videos and videos with comments disabled cannot be fully observed through the API.

## Before the demonstration: obtain API access

Each participant needs a Google account and a YouTube Data API v3 key. The usual setup is:

1. Open the [Google Cloud Console](https://console.cloud.google.com/) and create or select a project.
2. Open **APIs & Services → Library**.
3. Search for **YouTube Data API v3** and enable it.
4. Open **APIs & Services → Credentials**.
5. Select **Create credentials → API key**.
6. Keep the key available and enter it only when the notebook prompts you; do not save it in a public notebook.

For work beyond the classroom, restrict the key to the YouTube Data API v3 and review the credential restrictions available for the intended environment. A standard project has a daily quota, and every API method has an associated cost.

## API concepts used in this notebook

An API request calls a platform method with parameters such as a channel ID, video ID, date range, or page token. YouTube returns a structured JSON response. The notebook selects the required fields from that response and converts them into tables. A response contains only a limited number of items, so `nextPageToken` is used to request subsequent pages. These repeated requests consume part of the project's daily quota.

## 1. Install and import the required packages

Google Colab already includes `pandas`. The remaining packages provide access to the YouTube API and parse ISO 8601 video durations.

In [ ]:
%pip install -q google-api-python-client isodate

import json
import re
import shutil
import time
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

import isodate
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from IPython.display import display

## 2. Configure the extraction

Only this cell normally needs to be edited. `MAX_VIDEOS` and `MAX_COMMENTS_PER_VIDEO` keep the demonstration short. Set either value to `None` to remove that limit. Dates are interpreted in UTC and are inclusive.

In [ ]:
CHANNEL_ID = "UC-szrWWIUhm8Imy9s8fWHng"
CHANNEL_NAME = "Alan_Barroso"

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
MIN_VIDEO_DURATION_MINUTES = 8

MAX_VIDEOS = 5                 # Use None for all matching videos
MAX_COMMENTS_PER_VIDEO = 100  # Use None for all matching comments

SAVE_TO_DRIVE = True
OUTPUT_ROOT = "youtube_extraction_data"

START_DATETIME = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
END_DATETIME = (
    datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    + timedelta(days=1)
    - timedelta(microseconds=1)
)

if not CHANNEL_ID.startswith("UC"):
    raise ValueError("CHANNEL_ID must be a YouTube channel ID beginning with 'UC'.")
if START_DATETIME > END_DATETIME:
    raise ValueError("START_DATE must be earlier than or equal to END_DATE.")

## 3. Enter the API key securely

The `API_KEY` variable is intentionally left blank. When this cell runs, Colab asks for the key without displaying it. The value is used only during the current runtime and is not stored in the notebook. Never publish an API key in a notebook or GitHub repository.

In [ ]:
API_KEY = ""

if not API_KEY:
    API_KEY = getpass("Enter your YouTube Data API key: " ).strip()

if not API_KEY:
    raise ValueError("No API key was provided.")

youtube = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
print("YouTube API client created.")

## 4. Prepare the output folders

When `SAVE_TO_DRIVE` is `True`, the results are stored in Google Drive. Otherwise, they remain in Colab's temporary storage and disappear when the runtime is deleted.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    storage_root = Path("/content/drive/MyDrive")
else:
    storage_root = Path("/content")

channel_folder = re.sub(r"[^A-Za-z0-9_-]+", "_", CHANNEL_NAME).strip("_") or "channel"
period_folder = f"{START_DATE}_{END_DATE}"
OUTPUT_FOLDER = storage_root / OUTPUT_ROOT / channel_folder / period_folder
COMMENTS_BY_VIDEO_FOLDER = OUTPUT_FOLDER / "comments_by_video"
USERS_BY_VIDEO_FOLDER = OUTPUT_FOLDER / "users_by_video"

for folder in [OUTPUT_FOLDER, COMMENTS_BY_VIDEO_FOLDER, USERS_BY_VIDEO_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {OUTPUT_FOLDER}")

## 5. Define API and data-processing functions

YouTube returns paginated JSON responses. The functions below repeat requests while a `nextPageToken` is available, batch video-detail requests in groups of 50, retry temporary server errors, and stop explicitly if the daily API quota has been exhausted.

In [ ]:
def api_error_reason(error):
    try:
        payload = json.loads(error.content.decode("utf-8"))
        return payload["error"]["errors"][0].get("reason", "unknown")
    except Exception:
        return "unknown"


def execute_with_retries(request, max_retries=3):
    for attempt in range(max_retries):
        try:
            return request.execute()
        except HttpError as error:
            reason = api_error_reason(error)
            if reason in {"quotaExceeded", "dailyLimitExceeded"}:
                raise RuntimeError("The YouTube API quota has been exhausted.") from error
            if error.resp.status in {500, 502, 503, 504} and attempt < max_retries - 1:
                wait_seconds = 2 ** attempt
                print(f"Temporary API error. Retrying in {wait_seconds} second(s)...")
                time.sleep(wait_seconds)
                continue
            raise
    raise RuntimeError("The API request failed after several attempts.")


def parse_youtube_datetime(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def seconds_to_hhmmss(seconds):
    hours, remainder = divmod(int(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def get_uploads_playlist_id(channel_id):
    response = execute_with_retries(
        youtube.channels().list(part="snippet,contentDetails", id=channel_id)
    )
    if not response.get("items"):
        raise ValueError("The channel was not found. Check CHANNEL_ID.")
    channel = response["items"][0]
    playlist_id = channel["contentDetails"]["relatedPlaylists"]["uploads"]
    official_title = channel["snippet"]["title"]
    return playlist_id, official_title


def get_all_upload_video_ids(playlist_id):
    video_ids = []
    page_token = None
    while True:
        response = execute_with_retries(
            youtube.playlistItems().list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=page_token,
            )
        )
        video_ids.extend(
            item["contentDetails"]["videoId"]
            for item in response.get("items", [])
        )
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return video_ids


def get_video_details(video_ids):
    rows = []
    minimum_seconds = MIN_VIDEO_DURATION_MINUTES * 60
    for start in range(0, len(video_ids), 50):
        batch = video_ids[start:start + 50]
        response = execute_with_retries(
            youtube.videos().list(
                part="snippet,contentDetails,statistics",
                id=",".join(batch),
            )
        )
        for item in response.get("items", []):
            published_at = parse_youtube_datetime(item["snippet"]["publishedAt"])
            duration_seconds = int(
                isodate.parse_duration(item["contentDetails"]["duration"]).total_seconds()
            )
            if START_DATETIME <= published_at <= END_DATETIME and duration_seconds >= minimum_seconds:
                statistics = item.get("statistics", {})
                rows.append({
                    "video_id": item["id"],
                    "title": item["snippet"]["title"],
                    "published_at": item["snippet"]["publishedAt"],
                    "duration_seconds": duration_seconds,
                    "duration_hhmmss": seconds_to_hhmmss(duration_seconds),
                    "view_count": int(statistics["viewCount"]) if "viewCount" in statistics else None,
                    "comment_count_reported": int(statistics["commentCount"]) if "commentCount" in statistics else None,
                })
    return rows


def extract_top_level_comments(video_id, video_title):
    rows = []
    page_token = None
    try:
        while True:
            response = execute_with_retries(
                youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=page_token,
                    textFormat="plainText",
                    order="time",
                )
            )
            reached_start_date = False
            for thread in response.get("items", []):
                top_comment = thread["snippet"]["topLevelComment"]
                snippet = top_comment["snippet"]
                published_at = parse_youtube_datetime(snippet["publishedAt"])
                if published_at < START_DATETIME:
                    reached_start_date = True
                    break
                if published_at > END_DATETIME:
                    continue
                author_channel = snippet.get("authorChannelId") or {}
                rows.append({
                    "comment_id": top_comment["id"],
                    "video_id": video_id,
                    "video_title": video_title,
                    "author_channel_id": author_channel.get("value"),
                    "author_display_name": snippet.get("authorDisplayName"),
                    "text": snippet.get("textDisplay", ""),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "like_count": snippet.get("likeCount", 0),
                    "reply_count": thread["snippet"].get("totalReplyCount", 0),
                })
                if MAX_COMMENTS_PER_VIDEO is not None and len(rows) >= MAX_COMMENTS_PER_VIDEO:
                    return rows, "limit_reached"
            if reached_start_date:
                break
            page_token = response.get("nextPageToken")
            if not page_token:
                break
        return rows, "ok" if rows else "no_comments_in_period"
    except HttpError as error:
        reason = api_error_reason(error)
        status_map = {
            "commentsDisabled": "comments_disabled",
            "videoNotFound": "video_not_found",
            "forbidden": "video_unavailable",
        }
        return [], status_map.get(reason, f"api_error_{reason}")

## 6. Retrieve and filter the channel's videos

The uploads playlist is preferable to the API's search endpoint for this task because it provides a direct list of channel uploads and consumes fewer quota units. Video metadata are then requested in batches and filtered locally.

In [ ]:
uploads_playlist_id, official_channel_title = get_uploads_playlist_id(CHANNEL_ID)
print(f"Channel found: {official_channel_title}")

all_video_ids = get_all_upload_video_ids(uploads_playlist_id)
print(f"Uploads found in the channel: {len(all_video_ids):,}")

video_rows = get_video_details(all_video_ids)
videos_df = pd.DataFrame(video_rows)

if videos_df.empty:
    raise ValueError("No videos matched the selected dates and minimum duration.")

videos_df = videos_df.sort_values("published_at", ascending=False).reset_index(drop=True)
if MAX_VIDEOS is not None:
    videos_df = videos_df.head(MAX_VIDEOS).copy()

videos_path = OUTPUT_FOLDER / "videos.csv"
videos_df.to_csv(videos_path, index=False, encoding="utf-8-sig")

print(f"Videos selected for comment extraction: {len(videos_df):,}")
display(videos_df)

## 7. Extract comments and users

Each API response contains up to 100 comment threads. The code follows the pagination tokens until it reaches the selected limit, the beginning of the period, or the final page. Results are saved both as consolidated tables and as separate files for each video.

A displayed author name is not a stable identifier. When YouTube supplies it, `author_channel_id` is therefore retained alongside `author_display_name`. Some comments may not provide an author channel identifier.

In [ ]:
all_comment_rows = []
log_rows = []

for position, video in videos_df.iterrows():
    video_id = video["video_id"]
    video_title = video["title"]
    print(f"[{position + 1}/{len(videos_df)}] {video_title}")

    comment_rows, status = extract_top_level_comments(video_id, video_title)
    video_comments_df = pd.DataFrame(comment_rows)

    if not video_comments_df.empty:
        all_comment_rows.extend(comment_rows)
        video_comments_df.to_csv(
            COMMENTS_BY_VIDEO_FOLDER / f"{video_id}_comments.csv",
            index=False,
            encoding="utf-8-sig",
        )
        video_users_df = (
            video_comments_df[["author_channel_id", "author_display_name"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        video_users_df.to_csv(
            USERS_BY_VIDEO_FOLDER / f"{video_id}_users.csv",
            index=False,
            encoding="utf-8-sig",
        )

    log_rows.append({
        "video_id": video_id,
        "video_title": video_title,
        "status": status,
        "comments_extracted": len(comment_rows),
    })
    print(f"  Status: {status}; comments extracted: {len(comment_rows):,}")

comments_df = pd.DataFrame(all_comment_rows)
log_df = pd.DataFrame(log_rows)

if comments_df.empty:
    users_df = pd.DataFrame(columns=["author_channel_id", "author_display_name"])
else:
    users_df = (
        comments_df[["author_channel_id", "author_display_name"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

comments_df.to_csv(OUTPUT_FOLDER / "comments.csv", index=False, encoding="utf-8-sig")
users_df.to_csv(OUTPUT_FOLDER / "users.csv", index=False, encoding="utf-8-sig")
log_df.to_csv(OUTPUT_FOLDER / "extraction_log.csv", index=False, encoding="utf-8-sig")

print(f"Total top-level comments extracted: {len(comments_df):,}")
print(f"Distinct author records: {len(users_df):,}")
display(log_df)

## 8. Inspect and download the results

The preview helps verify the unit of analysis before continuing. Each row in `comments.csv` represents one top-level comment. The final cell creates a ZIP archive; in Colab it can also be downloaded directly.

In [ ]:
if not comments_df.empty:
    display(comments_df.head(10))
else:
    print("No comments were extracted. Check the extraction log.")

archive_path = shutil.make_archive(
    str(OUTPUT_FOLDER),
    "zip",
    root_dir=OUTPUT_FOLDER.parent,
    base_dir=OUTPUT_FOLDER.name,
)
print(f"ZIP archive created: {archive_path}")

try:
    from google.colab import files
    files.download(archive_path)
except Exception:
    print("Automatic download is available when the notebook runs in Google Colab.")

## Methodological notes

- API access does not make the platform fully observable. Deleted, private, restricted, or disabled content may be absent.
- The extraction date should be recorded because counts and availability can change.
- The API returns structured platform data, but the researcher still defines the sampling period, minimum duration, unit of analysis, and exclusion rules.
- This notebook filters comments by their publication date, not only by the video's publication date.
- Replies require additional `comments.list` requests and are not included here.
- Removing the demonstration limits can substantially increase runtime, stored data, and API-quota consumption.
- Public availability does not remove the need for data minimisation, secure storage, ethical review, and careful treatment of user identifiers.